# Gray σ = 50: the chroma-noise lift plus per-image R2R adaptation (pre-registered, amendment 5)

The gray lift (`gray-lift-kill-test`) puts the RGB prior above every supervised architecture without an external
prior on BSD68 and Set12, but 0.14–0.16 dB below PTG-RM (supervised + CLIP) at σ = 50. Per-image R2R gain tuning
(`adaptation-dev12`) added +0.127 dB at σ = 50 on Urban100's development images. This run combines the two, with
every setting frozen from Urban100 dev-12:

* **Parameters and training:** per-channel conv gains + GroupNorm γ/β; Muon (AdamW branch) at lr 1e-3; 90 steps of
  4 crops of ≤ 256².
* **R2R in the gray domain:** input y + ασz, target y − σz/α, with α = 0.5. The input is lifted to RGB with chroma
  noise at √3·σ√(1+α²) and embedded at the matching step; the output is the channel mean.
* **Test estimate:** lift ×2, one chroma draw plus its horizontal flip, at √3σ. The unadapted and adapted models see
  identical chroma draws.

Protocol: Restormer's gray script, seed 0 (the published draw). Success: adapted − unadapted ≥ +0.08 dB, Holm
sign-flip p < 0.01, ≤ 5 % of images worse by more than 0.05. SOTA: adapted mean ≥ PTG-RM + 0.05 (Set12 28.42,
BSD68 26.96). Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import json, math, os, queue, subprocess, sys, threading, time, traceback, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from PIL import Image

SIGMA, SEED = 50, 0
ALPHA, LR, CROP, BATCH, STEPS = 0.5, 1e-3, 256, 4, 90
PUBLISHED_PTG = {"Set12": 28.37, "BSD68": 26.91}
TIME_LIMIT_H = 11.5
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)

MUON_SRC = r'''"""Muon: momentum orthogonalised by Newton–Schulz, for hidden weight matrices.

Muon (Jordan et al., 2024) replaces the update of each weight matrix with the
nearest semi-orthogonal matrix to its Nesterov momentum, computed by five
quintic Newton–Schulz iterations; convolution kernels are flattened to
``(out, in · kh · kw)``. It is meant for hidden matrices only, so embeddings,
the output layer, biases and normalisation gains are updated by AdamW inside
the same optimizer, as in the reference implementation.

Muon updates are scaled by ``0.2 · sqrt(max(rows, columns))`` (Liu et al.,
2025, "Muon is Scalable for LLM Training"), which matches their RMS to AdamW's:
learning rates and weight decay tuned for AdamW carry over unchanged.
"""

from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn

NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


def muon_parameter_groups(model: nn.Module, adamw_modules: Iterable[str] = ("head", "lm_head")) -> list[dict]:
    """Split ``model``'s parameters into a Muon group and an AdamW group.

    AdamW gets every parameter with fewer than two dimensions, every
    ``nn.Embedding`` weight (with any output layer tied to it) and every
    parameter of a module named in ``adamw_modules``, matched against the
    qualified name or its last component. Muon gets the remaining matrices.
    """
    names = set(adamw_modules)
    adamw = {
        id(p)
        for name, module in model.named_modules()
        if isinstance(module, nn.Embedding) or name in names or name.rsplit(".", 1)[-1] in names
        for p in module.parameters(recurse=False)
    }
    muon, other = [], []
    for p in model.parameters():  # tied weights appear once
        (other if p.ndim < 2 or id(p) in adamw else muon).append(p)
    return [{"params": muon, "use_muon": True}, {"params": other, "use_muon": False}]


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices, AdamW for every other parameter.

    ``params`` holds tensors, split by shape (two or more dimensions go to
    Muon), or groups with a ``use_muon`` flag such as
    :func:`muon_parameter_groups` returns. ``lr`` and decoupled
    ``weight_decay`` apply to both kinds; ``momentum``, ``nesterov`` and
    ``ns_steps`` to Muon; ``betas`` and ``eps`` to AdamW.
    """

    def __init__(
        self,
        params,
        lr: float = 3e-4,
        weight_decay: float = 0.0,
        momentum: float = 0.95,
        nesterov: bool = True,
        ns_steps: int = 5,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
    ):
        params = list(params)
        if params and not isinstance(params[0], dict):
            params = [
                {"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                {"params": [p for p in params if p.ndim < 2], "use_muon": False},
            ]
        groups = [{**group, "params": list(group["params"])} for group in params]
        if any("use_muon" not in group for group in groups):
            raise ValueError("every parameter group needs a 'use_muon' flag")
        defaults = dict(
            lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, betas=betas, eps=eps,
        )
        super().__init__([group for group in groups if group["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected
'''  # laboratory/optimization/muon.py
_ns = {"__name__": "muon"}
exec(compile(MUON_SRC, "muon.py", "exec"), _ns)
Muon = _ns["Muon"]


def download(url, path, size=None):
    """Download with retries; no-op when the file is already complete."""
    if os.path.exists(path) and os.path.getsize(path) > 0 and (size is None or os.path.getsize(path) == size):
        return path
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path, "wb") as f:
                while chunk := r.read(1 << 23):
                    f.write(chunk)
            if size is None or os.path.getsize(path) == size:
                return path
        except Exception as e:
            print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"download failed: {url}")


t0 = time.time()
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
with ThreadPoolExecutor(8) as ex:
    adm_path = ex.submit(download, "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                         f"{CACHE}/adm.pt", 2_211_383_297)
    paths = {(ds, n): ex.submit(download, RAW.format(ds, n), f"{CACHE}/{ds}_{n}") for ds, ns in IMAGES.items() for n in ns}
    paths = {k: v.result() for k, v in paths.items()}
    adm_path = adm_path.result()
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TEST = {ds: [(n, np.asarray(Image.open(paths[(ds, n)]).convert("L"))) for n in ns] for ds, ns in IMAGES.items()}
TIMES["downloads"] = time.time() - t0
print({ds: len(v) for ds, v in TEST.items()}, f"downloads {TIMES['downloads'] / 60:.1f} min")


def noisy_of(clean_u8, sigma, seed):
    """Restormer's gray protocol: float32 / 255 plus np.random.normal(0, σ/255) after np.random.seed(seed), in place."""
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(clean_u8).to(a.device).double()
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

## 1 · ADM in fp32 weights with fp16 autocast and gradient checkpointing; per-channel conv gains + GroupNorm affine

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def checkpoint(func, inputs, params, flag):
    if flag and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(func, *inputs, use_reentrant=False)
    return func(*inputs)


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(2, 3).reshape(bs, -1, length)


gd_unet.checkpoint = checkpoint
gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(adm_path, map_location="cpu", weights_only=True)


def make_ctx(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=False,
                     use_scale_shift_norm=True, use_checkpoint=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False)
    gains = []
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            g = nn.Parameter(torch.ones(mod.out_channels, device=device))
            mod.register_forward_hook(lambda module, inp, out, g=g: out * g.view(1, -1, 1, 1).to(out.dtype))
            gains.append(g)
    norms = [p for mod in m.modules() if isinstance(mod, nn.GroupNorm) for p in (mod.weight, mod.bias)]
    for p in norms:
        p.requires_grad_(True)
    return dict(dev=device, adm=m, gains=gains, norms=norms, norms0=[p.detach().clone() for p in norms])


CTX = [make_ctx(d) for d in DEVICES]
ADM_STATE.clear()


@torch.no_grad()
def reset(ctx):
    for g in ctx["gains"]:
        g.fill_(1.0)
    for p, v in zip(ctx["norms"], ctx["norms0"]):
        p.copy_(v)

## 2 · The lifted one-step estimate and gray-domain R2R

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()
U = torch.full((3,), 1 / math.sqrt(3))


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def tweedie(model, y, sigma01):
    """One-step posterior mean of RGB y [B,3,H,W] ([0,1] units) at the step matched to sigma01, under fp16 autocast."""
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3]
    return (((x - math.sqrt(1 - ab) * eps.float()) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def chroma(shape, level01, gen, device):
    """w ~ N(0, 3·level²(I − uuᵀ)) per pixel, for a batch shape (B, H, W)."""
    z = torch.randn((shape[0], 3, *shape[1:]), generator=gen, device=device)
    u = U.to(device)[None, :, None, None]
    return math.sqrt(3) * level01 * (z - u * (u * z).sum(1, keepdim=True))


def lifted(model, y, level01, gen):
    """Gray y [B,H,W] with white noise at level01 -> channel mean of the posterior mean of its isotropic RGB lift."""
    Y = y[:, None].expand(-1, 3, -1, -1) + chroma(y.shape, level01, gen, y.device)
    return tweedie(model, Y, math.sqrt(3) * level01).mean(1)


@torch.no_grad()
def estimate(ctx, y, sigma, seed):
    """Lift ×2: one chroma draw on the image and the same draw on its mirror image (horizontal flip)."""
    gen = torch.Generator(device=ctx["dev"]).manual_seed(seed)
    a = lifted(ctx["adm"], y[None], sigma / 255, gen)[0]
    gen = torch.Generator(device=ctx["dev"]).manual_seed(seed)
    b = lifted(ctx["adm"], y.flip(-1)[None], sigma / 255, gen)[0].flip(-1)
    return a, (a + b) / 2


def crops(img, gen, n=BATCH, size=CROP):
    H, W = img.shape
    sh, sw = min(size, H), min(size, W)
    ii = torch.randint(0, H - sh + 1, (n,), generator=gen).tolist()
    jj = torch.randint(0, W - sw + 1, (n,), generator=gen).tolist()
    return torch.stack([img[i:i + sh, j:j + sw] for i, j in zip(ii, jj)])


def r2r_tune(ctx, noisy, sigma, seed):
    dev, s = ctx["dev"], sigma / 255
    opt = Muon(ctx["gains"] + ctx["norms"], lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    gen, ngen = torch.Generator().manual_seed(seed), torch.Generator(device=dev).manual_seed(77_000 + seed)
    for _ in range(STEPS):
        c = crops(noisy, gen)
        z = torch.randn(c.shape, generator=ngen, device=dev)
        loss = F.mse_loss(lifted(ctx["adm"], c + ALPHA * s * z, s * math.sqrt(1 + ALPHA ** 2), ngen), c - s * z / ALPHA)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

## 3 · Every image: unadapted, then adapted (identical noise and chroma draws)

In [ ]:
def run_job(ctx, job):
    ds, idx = job
    name, clean = TEST[ds][idx]
    y = noisy_of(clean, SIGMA, SEED).to(ctx["dev"])
    t0, seed = time.time(), 1000 * list(IMAGES).index(ds) + idx
    reset(ctx)
    x1, x2 = estimate(ctx, y, SIGMA, seed)
    rows = [dict(dataset=ds, image=name, adapted=False, flip=f, psnr=psnr(x, clean)) for f, x in (("×1", x1), ("×2", x2))]
    r2r_tune(ctx, y, SIGMA, seed)
    x1, x2 = estimate(ctx, y, SIGMA, seed)
    rows += [dict(dataset=ds, image=name, adapted=True, flip=f, psnr=psnr(x, clean)) for f, x in (("×1", x1), ("×2", x2))]
    reset(ctx)
    for r in rows:
        r["seconds"] = time.time() - t0
    return rows


def worker_pool(jobs, fn):
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 900:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
            pd.DataFrame([r for rows in results for r in rows]).to_csv(f"{OUT}/scores_partial.csv", index=False)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [("Set12", i) for i in range(12)] + [("BSD68", i) for i in range(68)]
res, fail = worker_pool(jobs, run_job)
TIMES["adaptation"] = time.time() - t0
SCORES = pd.DataFrame([r for rows in res for r in rows])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
print(f"{len(res)}/{len(jobs)} jobs, {len(fail)} failed, {TIMES['adaptation'] / 60:.1f} min")

## 4 · Pre-registered tests and the SOTA check

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())


rows = []
for ds, g in SCORES[SCORES.flip == "×2"].groupby("dataset"):
    w = g.pivot_table(index="image", columns="adapted", values="psnr")
    d = (w[True] - w[False]).to_numpy()
    rows.append(dict(dataset=ds, n=len(d), unadapted=w[False].mean(), adapted=w[True].mean(), gain=d.mean(),
                     worse_005=float((d < -0.05).mean()), p=sign_flip_p(d), ptg_rm=PUBLISHED_PTG[ds]))
T = pd.DataFrame(rows).sort_values("p")
T["p_holm"] = np.minimum(1.0, np.maximum.accumulate(T.p.to_numpy() * (len(T) - np.arange(len(T)))))
T["success"] = (T.gain >= 0.08) & (T.p_holm < 0.01) & (T.worse_005 <= 0.05)
T["sota"] = T.adapted >= T.ptg_rm + 0.05
print(T.round(4).to_string(index=False))
print("×1 means:", SCORES[SCORES.flip == "×1"].groupby(["dataset", "adapted"]).psnr.mean().round(3).to_dict())
T.to_csv(f"{OUT}/tests.csv", index=False)
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, tests=T.to_dict("records"), failed=[j for j, _ in fail]),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))